In [ ]:
# 1. Internet access is required for pip and for the git clone.#    Kaggle: Settings (right panel) -> Internet -> On, then restart the session.import sockettry:    socket.create_connection(("github.com", 443), timeout=10).close()    print("Internet: OK")except OSError as err:    print(f"Internet: FAILED ({err})")    print("Enable it in Settings -> Internet, then restart the session.")

In [ ]:
# 2. Secrets. Kaggle: Add Input -> Secrets -> New Secret.#    GITHUB_TOKEN is only needed while the repo is private (classic PAT, `repo` scope).#    Add a WANDB_API_KEY secret to log runs online; without it we log offline.import ostoken = os.environ.get("GITHUB_TOKEN", "").strip()username = "AKIF-jk"repo = "Generative_AI"branch = "task4"print(f"GITHUB_TOKEN found: {bool(token)}")

In [ ]:
# 3. Paths. Kaggle mounts attached datasets read-only under /kaggle/input and#    saves everything under /kaggle/working when you "Save Version".import subprocessWORK_DIR     = "/kaggle/working"REPO_DIR     = os.path.join(WORK_DIR, repo)INPUT_ROOT   = "/kaggle/input"EXTRACT_DIR  = os.path.join(WORK_DIR, "datasets")ARTIFACT_DIR = os.path.join(WORK_DIR, "artifacts")auth = f"{username}:{token}@" if token else ""REPO_URL = f"https://{auth}github.com/{username}/{repo}.git"for path in (WORK_DIR, EXTRACT_DIR, ARTIFACT_DIR):    os.makedirs(path, exist_ok=True)def git(*args, cwd=None):    # Only the subcommand is echoed, so the token never lands in a traceback.    proc = subprocess.run(["git", *args], cwd=cwd, text=True, capture_output=True)    if proc.returncode != 0:        raise RuntimeError(f"git {args[0]} failed:\n{proc.stderr.strip()}")    return proc.stdout.strip()print(f"Repo     : {REPO_DIR}")print(f"Inputs   : {INPUT_ROOT}")print(f"Artifacts: {ARTIFACT_DIR}")

In [ ]:
# 4. Clone the repository into /kaggle/workingif not os.path.isdir(os.path.join(REPO_DIR, ".git")):    git("clone", REPO_URL, REPO_DIR)# os.chdir persists across cells (like %cd), so the scripts keep finding# data/FS2K and best_generator.pth relative to the repo root.os.chdir(REPO_DIR)print(os.getcwd())

In [ ]:
# 5. Switch to the task4 branch, pull the latest, confirm the commitgit("switch", branch, cwd=REPO_DIR)git("pull", REPO_URL, branch, cwd=REPO_DIR)print(git("log", "-1", "--oneline", cwd=REPO_DIR))

In [ ]:
# 6. Safe to re-run: pull when the clone exists, clone otherwiseif os.path.isdir(os.path.join(REPO_DIR, ".git")):    git("pull", REPO_URL, branch, cwd=REPO_DIR)else:    git("clone", REPO_URL, REPO_DIR)os.chdir(REPO_DIR)print(git("status", "-sb", cwd=REPO_DIR))

In [ ]:
# 7. Add libraries here (requirements.txt mirrors the pinned ranges)!pip install torch!pip install -q -r requirements.txt

In [ ]:
# 8. Verify the GPU!nvidia-smiimport torchprint(f"GPU available: {torch.cuda.is_available()}")if torch.cuda.is_available():    print(f"GPU name:      {torch.cuda.get_device_name(0)}")    print(f"GPU count:     {torch.cuda.device_count()}")    print(f"Total VRAM:    {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")else:    print("WARNING: no GPU detected. Training will be extremely slow on CPU.")

In [ ]:
# 9. Weights & Biases - install!pip install wandb -qU

In [ ]:
# 10. Authenticate W&B (online with a secret, offline without one)import wandbif os.environ.get("WANDB_API_KEY"):    wandb.login()    print("WANDB_MODE: online")else:    os.environ["WANDB_MODE"] = "offline"    print("WANDB_MODE: offline (runs land in ./wandb/, upload with "          "`wandb sync wandb/offline-runs-*`)")

In [ ]:
# 11. Notebook-level run. train.py opens its own run under project "FS2K_Task4".wandb.init(    project="GenAI-Assignment1",    name="task4-face2sketch",    config={        "task": "Task4_Conditional_GAN_Face_to_Sketch",        "dataset": "FS2K",        "platform": "Kaggle",        "generator": "U-Net with style embedding",        "discriminator": "PatchGAN",        "loss_function": "LSGAN + L1 (lambda_l1=100)",        "hpo": "Optuna, 15 trials, 5 epochs each",        "framework": "PyTorch"    })if wandb.run and wandb.run.url:    print(f"W&B Run initialized: {wandb.run.url}")else:    print("W&B offline mode - logs are written to ./wandb/")config = wandb.configprint(f"Project: {config.task}")

In [ ]:
# 12. Locate FS2K.#     Attach the dataset with Add Input (a folder, or a .zip/.tar.gz of one) or#     upload it once as a private Kaggle dataset. Archives are extracted into#     /kaggle/working/datasets because /kaggle/input is read-only.import globimport tarfileimport zipfileos.chdir(REPO_DIR)def looks_like_fs2k(path):    return (os.path.isfile(os.path.join(path, "anno_train.json"))            and os.path.isdir(os.path.join(path, "photo"))            and os.path.isdir(os.path.join(path, "sketch")))def find_fs2k(root, max_depth=4):    found = []    if not os.path.isdir(root):        return found    for dirpath, dirnames, _ in os.walk(root):        if dirpath[len(root):].count(os.sep) >= max_depth:            dirnames[:] = []        if looks_like_fs2k(dirpath):            found.append(dirpath)            dirnames[:] = []    return founddef extract_archives():    for pattern in ("*.zip", "*.tar.gz", "*.tgz"):        archives = glob.glob(os.path.join(INPUT_ROOT, pattern))        archives += glob.glob(os.path.join(INPUT_ROOT, "*", pattern))        for archive in sorted(archives):            stem = os.path.basename(archive).split(".")[0]            target = os.path.join(EXTRACT_DIR, stem)            marker = target + ".extracted"            if os.path.exists(marker):                continue            os.makedirs(target, exist_ok=True)            print(f"Extracting {archive} -> {target}")            if archive.endswith(".zip"):                with zipfile.ZipFile(archive) as zf:                    zf.extractall(target)            else:                with tarfile.open(archive) as tf:                    tf.extractall(target)            open(marker, "w").close()extract_archives()candidates = find_fs2k(INPUT_ROOT) + find_fs2k(EXTRACT_DIR)if not candidates:    listing = sorted(os.listdir(INPUT_ROOT)) if os.path.isdir(INPUT_ROOT) else []    raise FileNotFoundError(        "Could not find FS2K.\n"        "Attach it with Add Input (folder or archive containing anno_train.json, "        "anno_test.json, photo/ and sketch/), or upload it as a private Kaggle "        "dataset and attach that.\n"        f"Attached inputs: {listing[:20]}\n"        f"Extracted archives: {sorted(os.listdir(EXTRACT_DIR))}"    )FS2K_SRC = candidates[0]print(f"FS2K source: {FS2K_SRC}")if len(candidates) > 1:    print(f"Other candidates ignored: {candidates[1:]}")

In [ ]:
# 13. Symlink the dataset as data/FS2K inside the repo - the scripts load#     "data/FS2K" relative to the repo root.DATA_DIR = os.path.join(REPO_DIR, "data", "FS2K")if os.path.islink(DATA_DIR):    os.remove(DATA_DIR)if os.path.isdir(DATA_DIR):    print(f"Using dataset already present at {DATA_DIR}")else:    os.makedirs(os.path.dirname(DATA_DIR), exist_ok=True)    os.symlink(FS2K_SRC, DATA_DIR)    print(f"Linked {DATA_DIR} -> {FS2K_SRC}")

In [ ]:
# 14. Verify the dataset layout and print the split sizes#     Expected: anno_train.json, anno_test.json, photo/, sketch/import jsonrequired = ["anno_train.json", "anno_test.json", "photo", "sketch"]missing = [r for r in required if not os.path.exists(os.path.join(DATA_DIR, r))]if missing:    raise FileNotFoundError(f"Incomplete FS2K dataset at {DATA_DIR}: missing {missing}")for split in ("train", "test"):    with open(os.path.join(DATA_DIR, f"anno_{split}.json")) as f:        anno = json.load(f)    counts = {}    for item in anno:        counts[item["style"]] = counts.get(item["style"], 0) + 1    print(f"{split:5s}: {len(anno):4d} pairs  |  "          + "  ".join(f"style {s}: {counts.get(s, 0)}" for s in sorted(counts)))print("\nValidation is a 15% stratified split of anno_train.json (seed 42).")print("anno_test.json is only read by evaluate.py, never during training or HPO.")

In [ ]:
# 15. Optional: config for this run. Set SMOKE_TEST = True for a 5-epoch check#     instead of the full 100-epoch schedule.#     A Kaggle GPU session is capped at 12 h, so shorten the schedule if needed.import reCONFIG = "src/task4_face2sketch/configs/task4.yaml"SMOKE_TEST = Falseif SMOKE_TEST:    text = open(CONFIG).read()    open(CONFIG, "w").write(re.sub(r"^epochs:.*$", "epochs: 5", text, flags=re.M))print(open(CONFIG).read())

In [ ]:
# 16. Optuna hyperparameter search.#     Searches lr_g, lr_d, batch_size, base_channels, dropout, style_dim,#     lambda_l1 over 15 trials with a median pruner and a 5-epoch budget.#     Objective: validation L1 (lower is better).!python src/task4_face2sketch/optuna_search.py

In [ ]:
# 17. Full training run!python src/task4_face2sketch/train.py --config {CONFIG}

In [ ]:
# 18. Checkpoint written by train.pyckpt = os.path.join(REPO_DIR, "best_generator.pth")size_mb = os.path.getsize(ckpt) / 1e6 if os.path.isfile(ckpt) else 0print(f"best_generator.pth: {'present' if size_mb else 'MISSING'} ({size_mb:.1f} MB)")# Note: evaluate.py and export_onnx.py build the model with base_channels=64# and style_dim=16. If the Optuna winner differs, update those defaults first.

In [ ]:
# 19. Evaluate best_generator.pth on the held-out test split.#     Reports per-style and overall L1 + SSIM and writes photo / target /#     generated / error-map panels into evaluation_results/.!python src/task4_face2sketch/evaluate.py

In [ ]:
# 20. Show the evaluation panelsimport globfrom IPython.display import Image, displaypanels = sorted(glob.glob(os.path.join(REPO_DIR, "evaluation_results", "*.png")))print(f"{len(panels)} panels in evaluation_results/")for panel in panels[:4]:    display(Image(filename=panel))

In [ ]:
# 21. Export the generator to ONNX and check PyTorch / ONNX Runtime parity (< 1e-4)!python src/task4_face2sketch/export_onnx.pyonnx_path = os.path.join(REPO_DIR, "generator.onnx")if os.path.isfile(onnx_path):    print(f"generator.onnx: {os.path.getsize(onnx_path) / 1e6:.1f} MB")else:    print("generator.onnx: MISSING")

In [ ]:
# 22. Collect the artifacts into /kaggle/working/artifacts so "Save Version"#     keeps them and the zip can be downloaded from the kernel output page.import shutilfor name in ("best_generator.pth", "generator.onnx"):    src = os.path.join(REPO_DIR, name)    if os.path.isfile(src):        shutil.copy2(src, ARTIFACT_DIR)eval_src = os.path.join(REPO_DIR, "evaluation_results")if os.path.isdir(eval_src):    shutil.copytree(eval_src, os.path.join(ARTIFACT_DIR, "evaluation_results"),                    dirs_exist_ok=True)for root, _, files in os.walk(ARTIFACT_DIR):    for name in files:        path = os.path.join(root, name)        print(f"{os.path.getsize(path) / 1e6:9.2f} MB  {os.path.relpath(path, ARTIFACT_DIR)}")

In [ ]:
# 23. Optional: push notebook changes back to the repo# Stage all changes# !git add .## Commit with a message# !git commit -m "task4: Kaggle results"## Push to the task4 branch# !git push {REPO_URL} {branch}

In [ ]:
# 24. Keep the token out of .git/config now that every git call used REPO_URLgit("remote", "set-url", "origin", f"https://github.com/{username}/{repo}.git",    cwd=REPO_DIR)print(git("remote", "get-url", "origin", cwd=REPO_DIR))